In [ ]:
import sys, warnings, logging
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.utils.config import DATA_PROCESSED, TARGET_COL, TIME_COL
from src.utils.plot_helpers import set_style, save_fig, PALETTE
from src.metrics.evaluation import evaluate_point, smape, mase, rmse
from src.models.cross_validation import run_cv
from src.models.baselines import (
    seasonal_naive_fn, theta_fn, prophet_fn, arima_fn
)
from src.models.forecast_store import save_forecasts, load_forecasts

set_style()

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_parquet("../data/processed/features_DE.parquet")
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).set_index(TIME_COL)
series = df[TARGET_COL].dropna()

print(f"Series length : {len(series):,}")
print(f"Range         : {series.index.min()} → {series.index.max()}")

# CV settings
N_SPLITS  = 5
HORIZON   = 24     # 24-hour ahead forecast
MIN_TRAIN = 24 * 7 * 8   # 8 weeks minimum training

In [ ]:
print("Running Seasonal Naive CV...")
cv_naive = run_cv(
    series, seasonal_naive_fn,
    n_splits=N_SPLITS, horizon=HORIZON,
    min_train_size=MIN_TRAIN,
    model_name="Seasonal Naive",
)
save_forecasts(cv_naive, "cv_seasonal_naive")
print(f"\nDone. Shape: {cv_naive.shape}")
cv_naive.head()

In [ ]:
print("Running Theta CV...")
cv_theta = run_cv(
    series, theta_fn,
    n_splits=N_SPLITS, horizon=HORIZON,
    min_train_size=MIN_TRAIN,
    model_name="Theta",
)
save_forecasts(cv_theta, "cv_theta")
print(f"\nDone. Shape: {cv_theta.shape}")

In [ ]:
print("Running Prophet CV (this takes ~2-3 min)...")
cv_prophet = run_cv(
    series, prophet_fn,
    n_splits=N_SPLITS, horizon=HORIZON,
    min_train_size=MIN_TRAIN,
    model_name="Prophet",
)
save_forecasts(cv_prophet, "cv_prophet")
print(f"\nDone. Shape: {cv_prophet.shape}")

In [ ]:
print("Running ARIMA CV (this takes ~3-5 min)...")
cv_arima = run_cv(
    series, arima_fn,
    n_splits=N_SPLITS, horizon=HORIZON,
    min_train_size=MIN_TRAIN,
    model_name="ARIMA",
)
save_forecasts(cv_arima, "cv_arima")
print(f"\nDone. Shape: {cv_arima.shape}")

In [ ]:
all_cv = pd.concat([cv_naive, cv_theta, cv_prophet, cv_arima],
                   ignore_index=True)

rows = []
for model_name, grp in all_cv.groupby("model"):
    rows.append(evaluate_point(
        actual=grp["actual"].values,
        forecast=grp["forecast"].values,
        label=model_name,
        seasonal_period=24,
    ))

scores = pd.DataFrame(rows).sort_values("smape")
scores = scores.round(3)
print("\n── Model Leaderboard ──────────────────────────────")
print(scores.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ["smape", "mase", "rmse"]
titles  = ["sMAPE (lower = better)",
           "MASE  (lower = better)",
           "RMSE  (lower = better)"]
colors  = [PALETTE["primary"], PALETTE["secondary"],
           PALETTE["accent"], PALETTE["warm"]]

for ax, metric, title in zip(axes, metrics, titles):
    vals   = scores.set_index("model")[metric]
    bars   = ax.barh(vals.index, vals.values,
                     color=colors[:len(vals)], alpha=0.85)
    ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=9)
    ax.set_title(title)
    ax.set_xlabel(metric.upper())
    ax.invert_yaxis()

plt.suptitle("Classical model comparison — CV scores", fontsize=13)
plt.tight_layout()
save_fig(fig, "10_model_leaderboard")
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=False)
model_results = {
    "Seasonal Naive": cv_naive,
    "Theta":          cv_theta,
    "Prophet":        cv_prophet,
    "ARIMA":          cv_arima,
}
plot_colors = [PALETTE["primary"], PALETTE["secondary"],
               PALETTE["accent"], PALETTE["warm"]]

for ax, (name, cv_df), color in zip(
    axes, model_results.items(), plot_colors
):
    # Use last fold only
    last_fold = cv_df[cv_df["fold"] == cv_df["fold"].max()]

    ax.plot(last_fold["timestamp"], last_fold["actual"],
            label="Actual", color="black", linewidth=1.5)
    ax.plot(last_fold["timestamp"], last_fold["forecast"],
            label=name, color=color, linewidth=1.5,
            linestyle="--")

    fold_smape = smape(last_fold["actual"].values,
                       last_fold["forecast"].values)
    ax.set_title(f"{name}  |  sMAPE = {fold_smape:.2f}%")
    ax.set_ylabel("Load (MW)")
    ax.legend(loc="upper right", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15)

plt.suptitle("Forecast vs actual — last CV fold (24h horizon)",
             fontsize=13)
plt.tight_layout()
save_fig(fig, "11_forecast_vs_actual")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for (name, color) in zip(
    ["Seasonal Naive","Theta","Prophet","ARIMA"],
    plot_colors
):
    grp = all_cv[all_cv["model"] == name]
    by_h = grp.groupby("horizon_h").apply(
        lambda g: smape(g["actual"].values, g["forecast"].values)
    ).reset_index()
    by_h.columns = ["horizon_h", "smape"]
    ax.plot(by_h["horizon_h"], by_h["smape"],
            label=name, color=color, linewidth=1.8, marker="o",
            markersize=3)

ax.set_xlabel("Forecast horizon (hours ahead)")
ax.set_ylabel("sMAPE (%)")
ax.set_title("sMAPE by forecast horizon — classical models")
ax.legend()
save_fig(fig, "12_smape_by_horizon")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, color) in zip(
    axes.flat,
    zip(["Seasonal Naive","Theta","Prophet","ARIMA"], plot_colors)
):
    grp = all_cv[all_cv["model"] == name]
    residuals = grp["actual"] - grp["forecast"]

    ax.hist(residuals, bins=40, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(0,                color="black", linewidth=1)
    ax.axvline(residuals.mean(), color=PALETTE["warm"],
               linewidth=1.5, linestyle="--",
               label=f"mean={residuals.mean():,.0f}")
    ax.set_title(f"{name} — residual distribution")
    ax.set_xlabel("Residual (MW)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.suptitle("Forecast residuals — all CV folds", fontsize=13)
plt.tight_layout()
save_fig(fig, "13_residual_distributions")
plt.show()

In [ ]:
# Save combined CV results for Phase 5 comparison
save_forecasts(all_cv, "cv_all_classical")

print("=" * 55)
print("PHASE 3 SUMMARY")
print("=" * 55)
print(f"CV folds    : {N_SPLITS}")
print(f"Horizon     : {HORIZON}h")
print(f"Total preds : {len(all_cv):,}")
print()
print("── Final Leaderboard ──────────────────────────────")
print(scores[["model","smape","mase","rmse"]].to_string(index=False))
print()

best = scores.iloc[0]["model"]
print(f"Best classical model : {best}")
print()
print("Saved to outputs/forecasts/:")
import os
for f in sorted(os.listdir("../outputs/forecasts")):
    print(f"  {f}")